# Dataset-3 Standalone Pothole Detection Training

This notebook trains and evaluates a YOLO detector independently on **Dataset-3** (665 images, 1,740 pothole annotations in YOLO format).

### Splits
- **Train**: 452 images
- **Val**: 80 images
- **Test**: 133 images


In [ ]:
# 1. Environment & Setup
from pathlib import Path
import torch
import yaml
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Dataset Paths and Hyperparameters
# When running locally from Source Codes/:
PROJECT_ROOT = Path("..").resolve()
DATA_YAML = PROJECT_ROOT / "Dataset-3_yolo" / "data.yaml"
OUTPUT_DIR = PROJECT_ROOT / "runs" / "dataset3_experiment"

# Hyperparameters
MODEL_SIZE = "s"          # 'n' (nano), 's' (small), 'm' (medium), or 'yolo11s'
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
PATIENCE = 15
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Data Config: {DATA_YAML}")
print(f"Output Dir: {OUTPUT_DIR}")


In [ ]:
# 3. Train YOLO Model
model = YOLO(f"yolov8{MODEL_SIZE}.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name=f"yolov8{MODEL_SIZE}_d3",
    patience=PATIENCE,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


In [ ]:
# 4. Helper Function to Display Evaluation Metrics
def print_metrics(metrics_dict, title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    metric_names = [
        ("Precision", "metrics/precision(B)"),
        ("Recall", "metrics/recall(B)"),
        ("mAP@0.5", "metrics/mAP50(B)"),
        ("mAP@0.5:0.95", "metrics/mAP50-95(B)"),
    ]
    for label, key in metric_names:
        val = metrics_dict.get(key)
        if val is not None:
            print(f"  {label:<18} {val:.4f} ({val*100:.2f}%)")
        else:
            print(f"  {label:<18} N/A")
    print("=" * 60 + "\n")

best_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"Best Weights: {best_weights}")
best_model = YOLO(str(best_weights))


In [ ]:
# 5. Validation Evaluation
val_results = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)
print_metrics(val_results.results_dict, "VALIDATION RESULTS — Dataset-3")


In [ ]:
# 6. Test Evaluation
test_results = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)
print_metrics(test_results.results_dict, "TEST RESULTS — Dataset-3")


In [ ]:
# 7. Qualitative Visualizations on Test Images
import matplotlib.pyplot as plt
import cv2
import glob

test_images = glob.glob(str(PROJECT_ROOT / "Dataset-3_yolo" / "images" / "test" / "*.jpg"))[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for ax, img_path in zip(axes.flatten(), test_images):
    preds = best_model.predict(img_path, conf=0.25, imgsz=IMG_SIZE, verbose=False)
    res_plot = preds[0].plot()
    ax.imshow(cv2.cvtColor(res_plot, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(img_path).name)
    ax.axis("off")

plt.tight_layout()
plt.show()
